# Notebook 04 — Verify UNSW publisher-upload against existing Kaggle mirror

**Step 6: Dataset source provenance, not model training.**

This notebook compares two CSV pairs (original publisher-upload candidates and the previous Kaggle mirror) using **progress bars** for file hashing, CSV loading, and cryptographic complete-row comparisons. It checks 10-class distributions, schemas, file byte hashes, all-column parsed-row ordering, full parsed-record multisets (duplicates preserved), and ID-independent equivalence if required.

**Provided publisher-upload paths (confirm files finish uploading):**
- `/kaggle/input/datasets/dsshayan/unsw-nb15/UNSW_NB15_training-set.csv`
- `/kaggle/input/datasets/dsshayan/unsw-nb15/UNSW_NB15_testing-set.csv`

**Existing mirror:** automatically resolves the correct official train/test roles and expected SHA-256 from our synced Step 3 manifest. The mirror's filenames are swapped.

**Scientific limit:** Matching content proves equivalence **between these uploaded files**, not independent authentication of where the files were obtained. Genuine chronological flow order remains unverified.

Enable **Internet** in Kaggle, enable access to Kaggle secret **`pushRare`**, and attach **both** datasets. No source records or individual signatures will be committed to GitHub.

In [ ]:
from pathlib import Path
import subprocess, sys, os

ORIGINAL_TRAIN = Path("/kaggle/input/datasets/dsshayan/unsw-nb15/UNSW_NB15_training-set.csv")
ORIGINAL_TEST  = Path("/kaggle/input/datasets/dsshayan/unsw-nb15/UNSW_NB15_testing-set.csv")
REPO = Path("/kaggle/working/rare-attack-ids-step06")
URL = "https://github.com/itsCodeBakery/rare-attack-spatiotemporal-ids.git"

print("Original training file:", ORIGINAL_TRAIN.is_file())
print("Original testing file:", ORIGINAL_TEST.is_file())
assert ORIGINAL_TRAIN.is_file() and ORIGINAL_TEST.is_file(), (
    "Wait for Kaggle dataset upload to finish and attach the dsshayan/unsw-nb15 dataset."
)

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", URL, str(REPO)], check=True)
print("Source commit:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())

In [ ]:
try:
    import tqdm
    print("tqdm progress bars:", tqdm.__version__)
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tqdm"], check=True)
    import tqdm
    print("tqdm installed:", tqdm.__version__)

MANIFEST = REPO / "data" / "manifests" / "unsw_nb15" / "audit_summary.json"
OUT = REPO / "data" / "manifests" / "unsw_nb15" / "publisher_comparison"
subprocess.run([
    sys.executable, str(REPO / "scripts" / "verify_unsw_source.py"),
    "--manifest", str(MANIFEST),
    "--official-train", str(ORIGINAL_TRAIN),
    "--official-test", str(ORIGINAL_TEST),
    "--output", str(OUT)
], cwd=str(REPO), check=True)
print((OUT / "source_comparison_report.md").read_text(encoding="utf-8"))

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("pushRare")
except Exception:
    token = None

if not token:
    print("Comparison completed but NOT SYNCED: allow access to Kaggle secret pushRare.")
else:
    env = os.environ.copy()
    env["GITHUB_TOKEN"] = token
    try:
        subprocess.run([
            sys.executable, str(REPO / "scripts" / "sync_source_comparison.py"),
            "--directory", str(OUT)
        ], cwd=str(REPO), env=env, check=True)
    finally:
        env.pop("GITHUB_TOKEN", None)
        del token

## Expected result

Read the **train/test outcomes** printed by the audit. Send the last-cell line beginning:

`VERIFIED SOURCE COMPARISON SYNC COMMIT:`

If content differs, don't rerun training yet. We'll inspect the aggregate report and determine whether serialization, IDs, row order, or real record changes are responsible.